# Direct Lyapunov Target-Selector Truth Test

This standalone diagnostic freezes the output disturbance estimate at zero and asks whether the linear target selector is a faithful target object for the nominal nonlinear polymer CSTR.

The test compares four quantities for each nominal setpoint:

- requested physical setpoint `y_sp`
- linear target-selector output `y_s`
- true nonlinear steady output at the linear selector's input `u_s`
- best reachable nonlinear steady output under the same input bounds

Existing experiment notebooks are intentionally not imported or modified.

In [ ]:
from utils.path_helpers import repo_path
import csv
import json
import os
from datetime import datetime
from pathlib import Path
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import least_squares, minimize

from Lyapunov.frozen_output_disturbance_target import solve_output_disturbance_target
from Simulation.system_functions import PolymerCSTR
from utils.direct_lyapunov_study import DIRECT_TWO_SETPOINT_Y_PHYS
from utils.scaling_helpers import apply_min_max, reverse_min_max
from utils.td3_helpers import load_and_prepare_system_data

In [ ]:
# Nominal polymer CSTR setup copied from the direct no-RL notebook.
Ad = 2.142e17
Ed = 14897
Ap = 3.816e10
Ep = 3557
At = 4.50e12
Et = 843
fi = 0.6
m_delta_H_r = -6.99e4
hA = 1.05e6
rhocp = 1506
rhoccpc = 4043
Mm = 104.14
system_params = np.array([Ad, Ed, Ap, Ep, At, Et, fi, m_delta_H_r, hA, rhocp, rhoccpc, Mm], dtype=float)

CIf = 0.5888
CMf = 8.6981
Qi = 108.0
Qs = 459.0
Tf = 330.0
Tcf = 295.0
V = 3000.0
Vc = 3312.4
system_design_params = np.array([CIf, CMf, Qi, Qs, Tf, Tcf, V, Vc], dtype=float)

system_steady_state_inputs = np.array([471.6, 378.0], dtype=float)  # [Qc, Qm]
delta_t = 0.5
u_min_phys = np.array([71.6, 78.0], dtype=float)
u_max_phys = np.array([870.0, 670.0], dtype=float)
y_sp_phys_all = DIRECT_TWO_SETPOINT_Y_PHYS.copy()

truth_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
truth_root = Path(os.fspath(repo_path())) / "results" / "target_selector_truth_test" / truth_timestamp
truth_root.mkdir(parents=True, exist_ok=True)

base_cstr = PolymerCSTR(
    system_params,
    system_design_params,
    system_steady_state_inputs,
    delta_t,
    deviation_form=False,
)
steady_states = {
    "ss_inputs": system_steady_state_inputs.copy(),
    "y_ss": base_cstr.y_ss.copy(),
}

system_data = load_and_prepare_system_data(
    steady_states=steady_states,
    setpoint_y=y_sp_phys_all,
    u_min=u_min_phys,
    u_max=u_max_phys,
    system_dict_path=os.path.join("Data", "system_dict"),
    augmentation_style="rawlings",
    augmentation_mode="output_disturbance",
)

A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]
data_min = system_data["data_min"]
data_max = system_data["data_max"]

n_u = int(B_aug.shape[1])
n_y = int(C_aug.shape[0])
n_x = int(A_aug.shape[0] - n_y)

u_ss_scaled = apply_min_max(steady_states["ss_inputs"], data_min[:n_u], data_max[:n_u])
u_min_scaled = apply_min_max(u_min_phys, data_min[:n_u], data_max[:n_u])
u_max_scaled = apply_min_max(u_max_phys, data_min[:n_u], data_max[:n_u])
u_dev_min = u_min_scaled - u_ss_scaled
u_dev_max = u_max_scaled - u_ss_scaled

y_ss_scaled = apply_min_max(steady_states["y_ss"], data_min[n_u:], data_max[n_u:])
y_sp_scaled_all = apply_min_max(y_sp_phys_all, data_min[n_u:], data_max[n_u:]) - y_ss_scaled
output_scale_phys = np.asarray(data_max[n_u:] - data_min[n_u:], dtype=float)

linear_target_config = {
    "solve_strategy": "lexicographic",
    "lexicographic_primary_tol_abs": 1.0e-10,
    "lexicographic_primary_tol_rel": 1.0e-8,
    "lexicographic_maxiter": 200,
    "lexicographic_ftol": 1.0e-10,
    "disturbance_model_mode": "output",
    "u_ref_weight": 0.0,
    "x_ref_weight": 0.0,
    "target_quality": {"enabled": False},
}

xhat_aug_zero_disturbance = np.zeros(n_x + n_y, dtype=float)

print("Truth-test output directory:", truth_root)
print("Nominal steady output:", steady_states["y_ss"])
print("Input bounds [Qc, Qm]:", u_min_phys, u_max_phys)
print("Frozen disturbance estimate d_hat_y:", xhat_aug_zero_disturbance[n_x:])

In [ ]:
def viscosity_from_state(x):
    x = np.asarray(x, dtype=float).reshape(-1)
    if x[-2] <= 0.0 or x[-1] <= 0.0:
        return np.nan
    return float(0.0012 * (Mm * x[-1] / x[-2]) ** 0.71)


def output_from_state(x):
    x = np.asarray(x, dtype=float).reshape(-1)
    return np.array([viscosity_from_state(x), x[2]], dtype=float)


def scaled_deviation_output(y_phys):
    return apply_min_max(y_phys, data_min[n_u:], data_max[n_u:]) - y_ss_scaled


def physical_output_from_scaled_deviation(y_dev):
    return reverse_min_max(np.asarray(y_dev, dtype=float) + y_ss_scaled, data_min[n_u:], data_max[n_u:])


def physical_input_from_dev(u_dev):
    u_scaled = np.asarray(u_dev, dtype=float).reshape(n_u) + u_ss_scaled
    return reverse_min_max(u_scaled, data_min[:n_u], data_max[:n_u])


def scaled_dev_from_physical_input(u_phys):
    return apply_min_max(u_phys, data_min[:n_u], data_max[:n_u]) - u_ss_scaled


def cstr_for_nominal_plant():
    return PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t, deviation_form=False)


state_lower = np.array([1.0e-9, 1.0e-9, 250.0, 250.0, 1.0e-12, 1.0e-12, 1.0e-9], dtype=float)
state_upper = np.array([2.0, 20.0, 450.0, 420.0, 10.0, 500.0, 1.0e6], dtype=float)
nominal_state = np.asarray(base_cstr.steady_trajectory, dtype=float).reshape(-1)


def _valid_state_guess(x):
    x = np.asarray(x, dtype=float).reshape(-1)
    return np.minimum(np.maximum(x, state_lower + 1.0e-10), state_upper - 1.0e-10)


def steady_residual_for_input(x, u_phys):
    cstr = cstr_for_nominal_plant()
    try:
        residual = np.asarray(cstr.odes(0.0, np.asarray(x, dtype=float), np.asarray(u_phys, dtype=float)), dtype=float)
    except Exception:
        return np.full(7, 1.0e8, dtype=float)
    if not np.all(np.isfinite(residual)):
        return np.full(7, 1.0e8, dtype=float)
    return residual


def solve_nonlinear_steady_for_input(u_phys, guesses=None):
    u_phys = np.asarray(u_phys, dtype=float).reshape(n_u)
    guesses = [] if guesses is None else list(guesses)
    candidate_guesses = [_valid_state_guess(nominal_state)]
    candidate_guesses.extend(_valid_state_guess(g) for g in guesses if g is not None)

    # Add a short rollout endpoint as a practical guess for off-nominal inputs.
    rollout = cstr_for_nominal_plant()
    rollout.current_input = u_phys.copy()
    for _ in range(250):
        rollout.step()
    candidate_guesses.append(_valid_state_guess(rollout.current_state))

    best = None
    for guess in candidate_guesses:
        result = least_squares(
            lambda x: steady_residual_for_input(x, u_phys),
            guess,
            bounds=(state_lower, state_upper),
            max_nfev=3000,
            xtol=1.0e-11,
            ftol=1.0e-11,
            gtol=1.0e-11,
        )
        residual_inf = float(np.max(np.abs(steady_residual_for_input(result.x, u_phys))))
        candidate = {
            "success": bool(result.success and residual_inf < 1.0e-6),
            "x_ss": np.asarray(result.x, dtype=float),
            "y_ss": output_from_state(result.x),
            "residual_inf": residual_inf,
            "cost": float(result.cost),
            "status": int(result.status),
            "message": str(result.message),
            "nfev": int(result.nfev),
        }
        if best is None or candidate["residual_inf"] < best["residual_inf"]:
            best = candidate
    return best


def scaled_inf_error(y_phys, y_sp_phys):
    err = (np.asarray(y_phys, dtype=float).reshape(n_y) - np.asarray(y_sp_phys, dtype=float).reshape(n_y)) / output_scale_phys
    return float(np.max(np.abs(err)))


def phys_inf_error(y_phys, y_sp_phys):
    return float(np.max(np.abs(np.asarray(y_phys, dtype=float).reshape(n_y) - np.asarray(y_sp_phys, dtype=float).reshape(n_y))))


def solve_best_nonlinear_bounded_target(y_sp_phys, seed_u_phys):
    y_sp_phys = np.asarray(y_sp_phys, dtype=float).reshape(n_y)
    seed_u_phys = np.minimum(np.maximum(np.asarray(seed_u_phys, dtype=float).reshape(n_u), u_min_phys), u_max_phys)
    cache = {}
    best_state_guess = None

    def eval_at_u(u_phys):
        nonlocal best_state_guess
        u_phys = np.minimum(np.maximum(np.asarray(u_phys, dtype=float).reshape(n_u), u_min_phys), u_max_phys)
        key = tuple(np.round(u_phys, 8))
        if key not in cache:
            guesses = [best_state_guess] if best_state_guess is not None else []
            ss = solve_nonlinear_steady_for_input(u_phys, guesses=guesses)
            if ss is not None and ss.get("success", False):
                best_state_guess = ss["x_ss"]
            cache[key] = ss
        return cache[key]

    def objective(u_phys):
        ss = eval_at_u(u_phys)
        if ss is None or not np.all(np.isfinite(ss["y_ss"])):
            return 1.0e12
        err = (ss["y_ss"] - y_sp_phys) / output_scale_phys
        penalty = 0.0 if ss.get("success", False) else 1.0e4 * max(ss.get("residual_inf", 1.0), 1.0)
        return float(err @ err + penalty)

    starts = [
        seed_u_phys,
        system_steady_state_inputs,
        0.5 * (u_min_phys + u_max_phys),
        u_min_phys,
        u_max_phys,
        np.array([u_min_phys[0], u_max_phys[1]], dtype=float),
        np.array([u_max_phys[0], u_min_phys[1]], dtype=float),
    ]
    best = None
    bounds = [(float(lo), float(hi)) for lo, hi in zip(u_min_phys, u_max_phys)]
    for start in starts:
        result = minimize(
            objective,
            np.minimum(np.maximum(np.asarray(start, dtype=float), u_min_phys), u_max_phys),
            method="L-BFGS-B",
            bounds=bounds,
            options={"maxiter": 80, "ftol": 1.0e-12},
        )
        ss = eval_at_u(result.x)
        value = objective(result.x)
        candidate = {
            "success": bool(result.success and ss is not None and ss.get("success", False)),
            "u_phys": np.asarray(result.x, dtype=float).reshape(n_u),
            "steady": ss,
            "objective": float(value),
            "optimizer_status": int(result.status),
            "optimizer_message": str(result.message),
            "optimizer_nit": int(result.nit),
        }
        if best is None or candidate["objective"] < best["objective"]:
            best = candidate
    return best

print("Helper functions ready.")

In [ ]:
linear_reachability_tol_scaled_inf = 0.05
nonlinear_reachability_tol_scaled_inf = 0.05
rows = []

for idx, (y_sp_phys, y_sp_scaled) in enumerate(zip(y_sp_phys_all, y_sp_scaled_all), start=1):
    linear_info = solve_output_disturbance_target(
        A_aug,
        B_aug,
        C_aug,
        xhat_aug_zero_disturbance,
        y_sp_scaled,
        target_mode="bounded",
        u_min=u_dev_min,
        u_max=u_dev_max,
        config=linear_target_config,
    )
    linear_success = bool(linear_info.get("success", False))
    linear_u_dev = np.asarray(linear_info.get("u_s", np.full(n_u, np.nan)), dtype=float).reshape(n_u)
    linear_u_phys = physical_input_from_dev(linear_u_dev)
    linear_y_scaled = np.asarray(linear_info.get("y_s", np.full(n_y, np.nan)), dtype=float).reshape(n_y)
    linear_y_phys = physical_output_from_scaled_deviation(linear_y_scaled)
    linear_gap_scaled = linear_y_scaled - y_sp_scaled
    linear_gap_phys = linear_y_phys - y_sp_phys
    linear_error_scaled_inf = float(np.max(np.abs(linear_gap_scaled)))
    linear_error_phys_inf = float(np.max(np.abs(linear_gap_phys)))

    nonlinear_at_linear = solve_nonlinear_steady_for_input(linear_u_phys)
    nonlinear_at_linear_y = nonlinear_at_linear["y_ss"]
    nonlinear_at_linear_scaled_inf = scaled_inf_error(nonlinear_at_linear_y, y_sp_phys)
    nonlinear_at_linear_phys_inf = phys_inf_error(nonlinear_at_linear_y, y_sp_phys)

    nonlinear_best = solve_best_nonlinear_bounded_target(y_sp_phys, linear_u_phys)
    nonlinear_best_u = nonlinear_best["u_phys"]
    nonlinear_best_y = nonlinear_best["steady"]["y_ss"]
    nonlinear_best_scaled_inf = scaled_inf_error(nonlinear_best_y, y_sp_phys)
    nonlinear_best_phys_inf = phys_inf_error(nonlinear_best_y, y_sp_phys)

    linear_reachable = bool(linear_success and linear_error_scaled_inf <= linear_reachability_tol_scaled_inf)
    nonlinear_reachable = bool(nonlinear_best["success"] and nonlinear_best_scaled_inf <= nonlinear_reachability_tol_scaled_inf)
    if not linear_success or nonlinear_best["steady"] is None:
        classification = "inconclusive_solver_failure"
    elif linear_reachable and nonlinear_reachable:
        classification = "linear_and_nonlinear_reachable"
    elif (not linear_reachable) and nonlinear_reachable:
        classification = "linear_says_unreachable_but_nonlinear_reachable"
    elif (not linear_reachable) and (not nonlinear_reachable):
        classification = "both_unreachable"
    else:
        classification = "linear_reachable_but_nonlinear_not"

    row = {
        "setpoint_index": idx,
        "classification": classification,
        "linear_success": linear_success,
        "linear_stage": linear_info.get("solve_stage"),
        "linear_bounded_solve_form": linear_info.get("bounded_solve_form"),
        "linear_lexicographic_stage": linear_info.get("lexicographic_stage"),
        "linear_exact_within_bounds": linear_info.get("exact_within_bounds"),
        "linear_active_lower_count": int(np.sum(np.asarray(linear_info.get("bounded_active_lower_mask", []), dtype=bool))),
        "linear_active_upper_count": int(np.sum(np.asarray(linear_info.get("bounded_active_upper_mask", []), dtype=bool))),
        "target_residual_total_norm": float(linear_info.get("residual_total_norm", np.nan)),
        "target_residual_out_norm": float(linear_info.get("residual_out_norm", np.nan)),
        "target_residual_dyn_norm": float(linear_info.get("residual_dyn_norm", np.nan)),
        "y_sp_eta": float(y_sp_phys[0]),
        "y_sp_T": float(y_sp_phys[1]),
        "linear_y_eta": float(linear_y_phys[0]),
        "linear_y_T": float(linear_y_phys[1]),
        "linear_gap_eta": float(linear_gap_phys[0]),
        "linear_gap_T": float(linear_gap_phys[1]),
        "linear_error_scaled_inf": linear_error_scaled_inf,
        "linear_error_phys_inf": linear_error_phys_inf,
        "linear_u_Qc": float(linear_u_phys[0]),
        "linear_u_Qm": float(linear_u_phys[1]),
        "nonlinear_at_linear_success": bool(nonlinear_at_linear.get("success", False)),
        "nonlinear_at_linear_residual_inf": float(nonlinear_at_linear.get("residual_inf", np.nan)),
        "nonlinear_at_linear_y_eta": float(nonlinear_at_linear_y[0]),
        "nonlinear_at_linear_y_T": float(nonlinear_at_linear_y[1]),
        "nonlinear_at_linear_error_scaled_inf": nonlinear_at_linear_scaled_inf,
        "nonlinear_at_linear_error_phys_inf": nonlinear_at_linear_phys_inf,
        "nonlinear_best_success": bool(nonlinear_best.get("success", False)),
        "nonlinear_best_optimizer_message": nonlinear_best.get("optimizer_message"),
        "nonlinear_best_residual_inf": float(nonlinear_best["steady"].get("residual_inf", np.nan)),
        "nonlinear_best_y_eta": float(nonlinear_best_y[0]),
        "nonlinear_best_y_T": float(nonlinear_best_y[1]),
        "nonlinear_best_error_scaled_inf": nonlinear_best_scaled_inf,
        "nonlinear_best_error_phys_inf": nonlinear_best_phys_inf,
        "nonlinear_best_u_Qc": float(nonlinear_best_u[0]),
        "nonlinear_best_u_Qm": float(nonlinear_best_u[1]),
    }
    rows.append(row)
    print(f"Setpoint {idx}: {classification}")
    pprint(row)

csv_path = truth_root / "target_selector_truth_test.csv"
with csv_path.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

json_path = truth_root / "target_selector_truth_test.json"
with json_path.open("w") as f:
    json.dump(rows, f, indent=2)

print("Saved:", csv_path)
print("Saved:", json_path)

In [ ]:
labels = [f"SP {row['setpoint_index']}" for row in rows]
output_series = [
    ("requested y_sp", np.array([[row["y_sp_eta"], row["y_sp_T"]] for row in rows], dtype=float)),
    ("linear y_s", np.array([[row["linear_y_eta"], row["linear_y_T"]] for row in rows], dtype=float)),
    ("nonlinear at linear u_s", np.array([[row["nonlinear_at_linear_y_eta"], row["nonlinear_at_linear_y_T"]] for row in rows], dtype=float)),
    ("best nonlinear", np.array([[row["nonlinear_best_y_eta"], row["nonlinear_best_y_T"]] for row in rows], dtype=float)),
]
colors = ["#111111", "#2a6f97", "#d1495b", "#2e7d32"]

x = np.arange(len(rows))
width = 0.18
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), dpi=160)
for out_idx, (ax, ylabel) in enumerate(zip(axes, ["eta", "T"])):
    for series_idx, (name, values) in enumerate(output_series):
        ax.bar(x + (series_idx - 1.5) * width, values[:, out_idx], width=width, label=name, color=colors[series_idx])
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].legend(frameon=False, fontsize=8)
fig.suptitle("Target-selector truth test: output comparison", y=1.02)
fig.tight_layout()
output_fig_path = truth_root / "target_selector_output_comparison.png"
fig.savefig(output_fig_path, bbox_inches="tight")
plt.show()

input_series = [
    ("linear u_s", np.array([[row["linear_u_Qc"], row["linear_u_Qm"]] for row in rows], dtype=float)),
    ("best nonlinear u", np.array([[row["nonlinear_best_u_Qc"], row["nonlinear_best_u_Qm"]] for row in rows], dtype=float)),
]
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), dpi=160)
for inp_idx, (ax, ylabel, lo, hi) in enumerate(zip(axes, ["Qc", "Qm"], u_min_phys, u_max_phys)):
    for series_idx, (name, values) in enumerate(input_series):
        ax.bar(x + (series_idx - 0.5) * 0.28, values[:, inp_idx], width=0.26, label=name, color=colors[series_idx + 1])
    ax.axhline(lo, color="#666666", linestyle="--", linewidth=1.0, label="bounds" if inp_idx == 0 else None)
    ax.axhline(hi, color="#666666", linestyle="--", linewidth=1.0)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].legend(frameon=False, fontsize=8)
fig.suptitle("Target-selector truth test: input comparison", y=1.02)
fig.tight_layout()
input_fig_path = truth_root / "target_selector_input_comparison.png"
fig.savefig(input_fig_path, bbox_inches="tight")
plt.show()

print("Saved figures:")
print(" ", output_fig_path)
print(" ", input_fig_path)

In [ ]:
classification_counts = {}
for row in rows:
    classification_counts[row["classification"]] = classification_counts.get(row["classification"], 0) + 1

print("Classification counts:")
pprint(classification_counts)

if classification_counts.get("linear_says_unreachable_but_nonlinear_reachable", 0) > 0:
    verdict = (
        "The linear target-selector model is the bottleneck for at least one nominal setpoint: "
        "the nonlinear plant can reach the setpoint within the configured tolerance, but the linear selector cannot."
    )
elif classification_counts.get("both_unreachable", 0) == len(rows):
    verdict = (
        "Both the linear target selector and the nonlinear bounded steady search classify all tested setpoints as unreachable. "
        "The main issue is setpoint admissibility under the stated input bounds."
    )
elif classification_counts.get("linear_and_nonlinear_reachable", 0) == len(rows):
    verdict = (
        "With d_hat_y frozen at zero, the linear target selector reaches all tested nominal setpoints. "
        "Closed-loop y_s-y_sp problems then point more strongly to observer disturbance drift or dynamic closed-loop effects."
    )
else:
    verdict = (
        "The truth test is mixed. Inspect the per-setpoint classifications and residuals before changing the controller."
    )

print("Final verdict:")
print(verdict)

with (truth_root / "verdict.txt").open("w") as f:
    f.write(verdict + "\n")